In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [12]:
!pip install -q onnx onnxscript optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 11.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 9.6 MB/s eta 0:00:00


In [8]:
import os
import json
import random
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Locate the official JSON files
# Note: Check the exact Kaggle input path, it is usually something like this:
base_dir = "/kaggle/input/datasets/mishalfatima09/facial-sketch-synthesis-dataset/FS2K" 

train_json_path = os.path.join(base_dir, "anno_train.json")
test_json_path = os.path.join(base_dir, "anno_test.json")

with open(train_json_path, 'r') as f:
    train_anno = json.load(f)

all_train_pairs = []
labels = []

def find_real_extension(base_path_without_ext):
    """Checks the disk and returns the path with the correct extension."""
    if os.path.exists(base_path_without_ext + '.jpg'):
        return base_path_without_ext + '.jpg'
    elif os.path.exists(base_path_without_ext + '.png'):
        return base_path_without_ext + '.png'
    elif os.path.exists(base_path_without_ext + '.JPG'):
        return base_path_without_ext + '.JPG'
    return base_path_without_ext + '.jpg' # Fallback
for item in train_anno:
    keys = list(item.keys())
    photo_key = next(k for k in keys if 'photo' in k.lower() or 'image' in k.lower())
    
    # Strip off any extension the JSON tries to give us, we will find it ourselves!
    raw_path = os.path.splitext(item[photo_key])[0] 
    
    # 1. Build Photo Path
    photo_rel = f"photo/{raw_path}" if not raw_path.startswith("photo/") else raw_path
    photo_base_path = os.path.join(base_dir, photo_rel)
    photo_path = find_real_extension(photo_base_path)
    
    # 2. Build Sketch Path 
    sketch_raw = raw_path.replace('photo', 'sketch').replace('image', 'sketch')
    sketch_rel = f"sketch/{sketch_raw}" if not sketch_raw.startswith("sketch/") else sketch_raw
    sketch_base_path = os.path.join(base_dir, sketch_rel)
    sketch_path = find_real_extension(sketch_base_path)
    # 3. Infer Style
    if 'photo1' in raw_path:
        style_idx = 0
    elif 'photo2' in raw_path:
        style_idx = 1
    else:
        style_idx = 2
        
    all_train_pairs.append((photo_path, sketch_path, style_idx))
    labels.append(style_idx)
    
# 2. Stratified Validation Split (15%, Random Seed 42, Stratified by style)
train_data, val_data = train_test_split(
    all_train_pairs, 
    test_size=0.15, 
    random_state=42, 
    stratify=labels
)

# 3. Custom Dataset with IDENTICAL Spatial Augmentations
class FS2KDataset(Dataset):
    def __init__(self, data_list, is_train=True):
        self.data_list = data_list
        self.is_train = is_train

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        photo_path, sketch_path, style_label = self.data_list[idx]
        
        photo = Image.open(photo_path).convert('RGB')
        sketch = Image.open(sketch_path).convert('RGB')

        # Resize to 128x128
        photo = TF.resize(photo, (128, 128))
        sketch = TF.resize(sketch, (128, 128))

        # IDENTICAL augmentations (Both images must undergo the exact same transform!)
        if self.is_train:
            # Random Horizontal Flip
            if random.random() > 0.5:
                photo = TF.hflip(photo)
                sketch = TF.hflip(sketch)
                
            # Random slight rotation
            if random.random() > 0.5:
                angle = random.uniform(-10, 10)
                photo = TF.rotate(photo, angle)
                sketch = TF.rotate(sketch, angle)

        # Convert to Tensor and Normalize to [-1, 1] for GANs
        photo = TF.to_tensor(photo)
        sketch = TF.to_tensor(sketch)
        
        photo = TF.normalize(photo, [0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        sketch = TF.normalize(sketch, [0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

        return photo, sketch, torch.tensor(style_label, dtype=torch.long)

# 4. Initialize DataLoaders
train_loader = DataLoader(FS2KDataset(train_data, is_train=True), batch_size=16, shuffle=True)
val_loader = DataLoader(FS2KDataset(val_data, is_train=False), batch_size=16, shuffle=False)

print(f"Loaded {len(train_data)} training pairs and {len(val_data)} validation pairs.")
p, s, label = next(iter(train_loader))
print(f"Batch Photo shape: {p.shape}, Sketch shape: {s.shape}, Labels: {label.shape}")

Using device: cuda
Loaded 899 training pairs and 159 validation pairs.
Batch Photo shape: torch.Size([16, 3, 128, 128]), Sketch shape: torch.Size([16, 3, 128, 128]), Labels: torch.Size([16])


In [9]:
import torch.nn as nn

class Block(nn.Module):
    def __init__(self, in_c, out_c, down=True, act="relu", drop=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False) if down else nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_c, affine=True),
            nn.ReLU(inplace=True) if act == "relu" else nn.LeakyReLU(0.2, inplace=True)
        )
        self.drop = nn.Dropout(drop) if drop > 0.0 else nn.Identity()

    def forward(self, x):
        return self.drop(self.conv(x))

class GeneratorUNet(nn.Module):
    def __init__(self, embed_dim=16, base_c=64, drop_rate=0.5):
        super().__init__()
        self.embed = nn.Embedding(3, embed_dim)
        
        self.down1 = Block(3 + embed_dim, base_c, down=True, act="leaky") 
        self.down2 = Block(base_c, base_c*2, down=True, act="leaky") 
        self.down3 = Block(base_c*2, base_c*4, down=True, act="leaky") 
        self.down4 = Block(base_c*4, base_c*8, down=True, act="leaky") 
        
        self.up1 = Block(base_c*8, base_c*4, down=False, act="relu", drop=drop_rate)
        self.up2 = Block(base_c*8, base_c*2, down=False, act="relu", drop=drop_rate)
        self.up3 = Block(base_c*4, base_c, down=False, act="relu")
        
        self.final = nn.Sequential(nn.ConvTranspose2d(base_c*2, 3, 4, 2, 1), nn.Tanh())

    def forward(self, photo, style):
        # Expand style embedding to match image size
        s_emb = self.embed(style).view(-1, self.embed.embedding_dim, 1, 1)
        s_emb = s_emb.expand(-1, -1, photo.shape[2], photo.shape[3])
        x_in = torch.cat([photo, s_emb], dim=1)
        
        d1 = self.down1(x_in)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        
        # U-Net Skip Connections
        u1 = self.up1(d4)
        u2 = self.up2(torch.cat([u1, d3], dim=1))
        u3 = self.up3(torch.cat([u2, d2], dim=1))
        return self.final(torch.cat([u3, d1], dim=1))

class PatchGANDiscriminator(nn.Module):
    def __init__(self, embed_dim=16, base_c=64):
        super().__init__()
        self.embed = nn.Embedding(3, embed_dim)
        self.model = nn.Sequential(
            nn.Conv2d(6 + embed_dim, base_c, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_c, base_c*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(base_c*2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_c*2, base_c*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(base_c*4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_c*4, 1, 4, 1, 1) # Output 1-channel Patch Grid
        )

    def forward(self, photo, sketch, style):
        s_emb = self.embed(style).view(-1, self.embed.embedding_dim, 1, 1)
        s_emb = s_emb.expand(-1, -1, photo.shape[2], photo.shape[3])
        x_in = torch.cat([photo, sketch, s_emb], dim=1)
        return self.model(x_in)

In [10]:
import optuna
import torch.optim as optim

bce_loss = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()

def cgan_objective(trial):
    # PDF Requirements: Tune these parameters
    lr_g = trial.suggest_float("lr_g", 1e-4, 1e-3, log=True)
    lr_d = trial.suggest_float("lr_d", 1e-5, 1e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    base_c = trial.suggest_categorical("base_c", [32, 64])
    drop_rate = trial.suggest_float("drop_rate", 0.0, 0.5)
    embed_dim = trial.suggest_categorical("embed_dim", [8, 16, 32])
    lambda_l1 = trial.suggest_float("lambda_l1", 50, 150)
    
    # Dynamic Dataloader for batch size
    t_loader = DataLoader(FS2KDataset(train_data, is_train=True), batch_size=batch_size, shuffle=True)
    
    gen = GeneratorUNet(embed_dim=embed_dim, base_c=base_c, drop_rate=drop_rate).to(device)
    disc = PatchGANDiscriminator(embed_dim=embed_dim, base_c=base_c).to(device)
    
    opt_g = optim.Adam(gen.parameters(), lr=lr_g, betas=(0.5, 0.999))
    opt_d = optim.Adam(disc.parameters(), lr=lr_d, betas=(0.5, 0.999))
    
    # Train 1 fast epoch just to evaluate for Optuna
    gen.train()
    disc.train()
    for photo, real_sketch, style in t_loader:
        photo, real_sketch, style = photo.to(device), real_sketch.to(device), style.to(device)
        
        # --- Train Discriminator ---
        opt_d.zero_grad()
        fake_sketch = gen(photo, style)
        
        pred_real = disc(photo, real_sketch, style)
        loss_d_real = bce_loss(pred_real, torch.ones_like(pred_real))
        
        pred_fake = disc(photo, fake_sketch.detach(), style)
        loss_d_fake = bce_loss(pred_fake, torch.zeros_like(pred_fake))
        
        loss_d = (loss_d_real + loss_d_fake) * 0.5
        loss_d.backward()
        opt_d.step()
        
        # --- Train Generator ---
        opt_g.zero_grad()
        pred_fake_g = disc(photo, fake_sketch, style)
        loss_g_adv = bce_loss(pred_fake_g, torch.ones_like(pred_fake_g))
        loss_g_l1 = l1_loss(fake_sketch, real_sketch) * lambda_l1
        
        loss_g = loss_g_adv + loss_g_l1
        loss_g.backward()
        opt_g.step()
        
    return loss_g.item()

print("Starting Fast GAN Optuna Search...")
cgan_study = optuna.create_study(direction="minimize")
cgan_study.optimize(cgan_objective, n_trials=3) # Just 3 fast trials!

print(f"Best Trial: {cgan_study.best_trial.value}")
print("Best Params:", cgan_study.best_trial.params)

[I 2026-10-04 12:27:50,123] A new study created in memory with name: no-name-54eb887d-0840-4ed2-af6d-4e9b68838cd1


Starting Fast GAN Optuna Search...


[I 2026-10-04 12:28:26,086] Trial 0 finished with value: 46.92245101928711 and parameters: {'lr_g': 0.00015548263228315238, 'lr_d': 3.0092288301207088e-05, 'batch_size': 16, 'base_c': 64, 'drop_rate': 0.10409313806768716, 'embed_dim': 32, 'lambda_l1': 121.19840304345652}. Best is trial 0 with value: 46.92245101928711.
[I 2026-10-04 12:28:35,158] Trial 1 finished with value: 17.834901809692383 and parameters: {'lr_g': 0.00029312904983195854, 'lr_d': 4.362605332652167e-05, 'batch_size': 32, 'base_c': 64, 'drop_rate': 0.20230205011528596, 'embed_dim': 8, 'lambda_l1': 54.21126585837458}. Best is trial 1 with value: 17.834901809692383.
[I 2026-10-04 12:28:43,102] Trial 2 finished with value: 36.97819137573242 and parameters: {'lr_g': 0.0004427279070539511, 'lr_d': 6.36840335933603e-05, 'batch_size': 16, 'base_c': 64, 'drop_rate': 0.27668334045421855, 'embed_dim': 8, 'lambda_l1': 144.17838572753564}. Best is trial 1 with value: 17.834901809692383.


Best Trial: 17.834901809692383
Best Params: {'lr_g': 0.00029312904983195854, 'lr_d': 4.362605332652167e-05, 'batch_size': 32, 'base_c': 64, 'drop_rate': 0.20230205011528596, 'embed_dim': 8, 'lambda_l1': 54.21126585837458}


In [13]:
import torchvision.utils as vutils

# 1. Grab the winning hyperparameters from Optuna
best_p = cgan_study.best_trial.params
final_batch = best_p["batch_size"]

# Re-init dataloaders with the winning batch size
train_loader = DataLoader(FS2KDataset(train_data, is_train=True), batch_size=final_batch, shuffle=True)
val_loader = DataLoader(FS2KDataset(val_data, is_train=False), batch_size=final_batch, shuffle=False)

# 2. Build final networks
gen = GeneratorUNet(embed_dim=best_p["embed_dim"], base_c=best_p["base_c"], drop_rate=best_p["drop_rate"]).to(device)
disc = PatchGANDiscriminator(embed_dim=best_p["embed_dim"], base_c=best_p["base_c"]).to(device)

opt_g = optim.Adam(gen.parameters(), lr=best_p["lr_g"], betas=(0.5, 0.999))
opt_d = optim.Adam(disc.parameters(), lr=best_p["lr_d"], betas=(0.5, 0.999))

# 3. Final Training Loop (Tracking exact metrics required by PDF)
print("--- STARTING FINAL CGAN TRAINING ---")
num_epochs = 3 # I set this to 3 so you can finish quickly! (You can increase this if you want higher quality)

for epoch in range(num_epochs):
    gen.train()
    disc.train()
    
    # Trackers required by PDF
    epoch_d_real, epoch_d_fake = 0.0, 0.0
    epoch_g_adv, epoch_g_recon = 0.0, 0.0
    
    for photo, real_sketch, style in train_loader:
        photo, real_sketch, style = photo.to(device), real_sketch.to(device), style.to(device)
        
        # -- Train Discriminator --
        opt_d.zero_grad()
        fake_sketch = gen(photo, style)
        
        pred_real = disc(photo, real_sketch, style)
        loss_d_real = bce_loss(pred_real, torch.ones_like(pred_real))
        
        pred_fake = disc(photo, fake_sketch.detach(), style) # detach() so we don't train G here
        loss_d_fake = bce_loss(pred_fake, torch.zeros_like(pred_fake))
        
        loss_d = (loss_d_real + loss_d_fake) * 0.5
        loss_d.backward()
        opt_d.step()
        
        # -- Train Generator --
        opt_g.zero_grad()
        pred_fake_g = disc(photo, fake_sketch, style)
        loss_g_adv = bce_loss(pred_fake_g, torch.ones_like(pred_fake_g))
        loss_g_l1 = l1_loss(fake_sketch, real_sketch) * best_p["lambda_l1"]
        
        loss_g = loss_g_adv + loss_g_l1
        loss_g.backward()
        opt_g.step()
        
        # Update trackers
        epoch_d_real += loss_d_real.item()
        epoch_d_fake += loss_d_fake.item()
        epoch_g_adv += loss_g_adv.item()
        epoch_g_recon += loss_g_l1.item()
        
    batches = len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] | D_Real: {epoch_d_real/batches:.4f} | D_Fake: {epoch_d_fake/batches:.4f} | G_Adv: {epoch_g_adv/batches:.4f} | G_Recon: {epoch_g_recon/batches:.4f}")

# 4. Export the final generator to ONNX
print("\nExporting Generator to ONNX...")
gen.eval()
dummy_photo = torch.randn(1, 3, 128, 128).to(device)
dummy_style = torch.tensor([0], dtype=torch.long).to(device) # Just test with Style 1 (0-indexed)

onnx_name = "cgan_generator.onnx"
torch.onnx.export(
    gen, (dummy_photo, dummy_style), onnx_name,
    export_params=True, opset_version=18, do_constant_folding=True,
    input_names=['photo', 'style'], output_names=['sketch'],
    dynamic_axes={'photo': {0: 'batch_size'}, 'style': {0: 'batch_size'}, 'sketch': {0: 'batch_size'}}
)
print(f"SUCCESS! The Generator has been exported to '{onnx_name}'!!")

--- STARTING FINAL CGAN TRAINING ---
Epoch [1/3] | D_Real: 0.6177 | D_Fake: 0.6080 | G_Adv: 0.8395 | G_Recon: 26.8338
Epoch [2/3] | D_Real: 0.6083 | D_Fake: 0.5926 | G_Adv: 0.8793 | G_Recon: 16.3485
Epoch [3/3] | D_Real: 0.5583 | D_Fake: 0.5549 | G_Adv: 0.9189 | G_Recon: 15.2828

Exporting Generator to ONNX...


/tmp/ipykernel_49/1801991830.py:73: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `GeneratorUNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GeneratorUNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
SUCCESS! The Generator has been exported to 'cgan_generator.onnx'!!


/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
